In [1]:
import pandas as pd
import numpy as np
import glob

In [3]:
files = glob.glob("../data/raw/*.csv")

print(files)

['../data/raw/LAX_Jetblue.csv', '../data/raw/SFO_Alaskan.csv', '../data/raw/SFO_Delta.csv', '../data/raw/SAN_Jetblue.csv', '../data/raw/SAN_Delta.csv', '../data/raw/LAX_AA.csv', '../data/raw/SFO_AA.csv', '../data/raw/SAN_AA.csv', '../data/raw/LAX_United.csv', '../data/raw/SAN_Alaskan.csv', '../data/raw/SAN_United.csv', '../data/raw/LAX_Delta.csv', '../data/raw/SFO_Jetblue.csv', '../data/raw/LAX_Alaskan.csv', '../data/raw/SFO_United.csv']


In [5]:
dfs = []

for file in files:
    df = pd.read_csv(file, skiprows = 6)
    dfs.append(df)

flights = pd.concat(dfs, ignore_index = True)
print(flights.shape)

(287401, 17)


In [6]:
flights.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,B6,01/01/2025,100.0,N2188J,FLL,10:43,10:34,302.0,282.0,-9.0,10:48,14.0,0.0,0.0,0.0,0.0,0.0
1,B6,01/01/2025,200.0,N982JB,FLL,23:05,23:12,291.0,288.0,7.0,23:37,25.0,0.0,0.0,0.0,0.0,0.0
2,B6,01/01/2025,288.0,N2156J,BOS,11:05,10:52,325.0,316.0,-13.0,11:10,18.0,0.0,0.0,0.0,0.0,0.0
3,B6,01/01/2025,388.0,N2180J,BOS,20:39,20:32,322.0,298.0,-7.0,20:44,12.0,0.0,0.0,0.0,0.0,0.0
4,B6,01/01/2025,424.0,N2165J,JFK,06:35,06:21,324.0,296.0,-14.0,06:42,21.0,0.0,0.0,0.0,0.0,0.0


In [7]:
flights.columns

Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Wheels-off time', 'Taxi-Out time (Minutes)', 'Delay Carrier (Minutes)',
       'Delay Weather (Minutes)', 'Delay National Aviation System (Minutes)',
       'Delay Security (Minutes)', 'Delay Late Aircraft Arrival (Minutes)'],
      dtype='object')

In [9]:
flights.columns =(
    flights.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex = False)
    .str.replace(")", "", regex = False)
    .str.replace("-", "_")
)
flights.columns

Index(['carrier_code', 'date_mm/dd/yyyy', 'flight_number', 'tail_number',
       'destination_airport', 'scheduled_departure_time',
       'actual_departure_time', 'scheduled_elapsed_time_minutes',
       'actual_elapsed_time_minutes', 'departure_delay_minutes',
       'wheels_off_time', 'taxi_out_time_minutes', 'delay_carrier_minutes',
       'delay_weather_minutes', 'delay_national_aviation_system_minutes',
       'delay_security_minutes', 'delay_late_aircraft_arrival_minutes'],
      dtype='object')

In [10]:
print(flights.shape)
flights.isnull().sum()

(287401, 17)


carrier_code                                0
date_mm/dd/yyyy                            15
flight_number                              15
tail_number                               639
destination_airport                        15
scheduled_departure_time                   15
actual_departure_time                      15
scheduled_elapsed_time_minutes             15
actual_elapsed_time_minutes                15
departure_delay_minutes                    15
wheels_off_time                            15
taxi_out_time_minutes                      15
delay_carrier_minutes                      15
delay_weather_minutes                      15
delay_national_aviation_system_minutes     15
delay_security_minutes                     15
delay_late_aircraft_arrival_minutes        15
dtype: int64

In [11]:
flights = flights.dropna(
    subset=[
        "date_mm/dd/yyyy",
        "scheduled_departure_time",
        "departure_delay_minutes"
    ]
)

print(flights.shape)

(287386, 17)


In [12]:
flights["date_mm/dd/yyyy"] = pd.to_datetime(flights["date_mm/dd/yyyy"])

flights["month"] = (flights["date_mm/dd/yyyy"].dt.month)

flights["day_of_week"] = (flights["date_mm/dd/yyyy"].dt.day_name())

flights["departure_hour"] = (
    flights["scheduled_departure_time"]
    .astype(str)
    .str.split(":")
    .str[0]
    .astype(int)
)

flights["delayed_15"] = (flights["departure_delay_minutes"] >= 15).astype(int)

In [15]:
flights.head()

,carrier_code,date_mm/dd/yyyy,flight_number,tail_number,destination_airport,scheduled_departure_time,actual_departure_time,scheduled_elapsed_time_minutes,actual_elapsed_time_minutes,departure_delay_minutes,...,taxi_out_time_minutes,delay_carrier_minutes,delay_weather_minutes,delay_national_aviation_system_minutes,delay_security_minutes,delay_late_aircraft_arrival_minutes,month,day_of_week,departure_hour,delayed_15
0,B6,2025-01-01,100.0,N2188J,FLL,10:43,10:34,302.0,282.0,-9.0,...,14.0,0.0,0.0,0.0,0.0,0.0,1,Wednesday,10,0
1,B6,2025-01-01,200.0,N982JB,FLL,23:05,23:12,291.0,288.0,7.0,...,25.0,0.0,0.0,0.0,0.0,0.0,1,Wednesday,23,0
2,B6,2025-01-01,288.0,N2156J,BOS,11:05,10:52,325.0,316.0,-13.0,...,18.0,0.0,0.0,0.0,0.0,0.0,1,Wednesday,11,0
3,B6,2025-01-01,388.0,N2180J,BOS,20:39,20:32,322.0,298.0,-7.0,...,12.0,0.0,0.0,0.0,0.0,0.0,1,Wednesday,20,0
4,B6,2025-01-01,424.0,N2165J,JFK,06:35,06:21,324.0,296.0,-14.0,...,21.0,0.0,0.0,0.0,0.0,0.0,1,Wednesday,6,0
